# Optimisation Énergétique - Modélisation par Graphe Temporel

## Nouvelle approche: Graphe (r,t)

Cette version utilise un **graphe temporel** où chaque nœud représente un état (rénovation r) à un instant t.

### Concepts clés:
- **Nœuds**: Tuples (r, t) représentant l'état r au mois t
- **Arêtes d'attente**: (r, t) → (r, t+1) - rester dans le même état
- **Arêtes de transition**: (r, t) → (r', t+d) - passer de r à r' (durée d mois)
- **Variable de décision**: f[b, e] = flux sur l'arête e du bâtiment b
- **Contrainte de flux**: Conservation du flux en chaque nœud

### Avantages attendus:
1. Contraintes de flux locales (par nœud) au lieu de cumulatives
2. Structure plus naturelle du problème
3. Meilleure compréhension des chemins suivis
4. Potentiellement moins de contraintes redondantes

# Importations

In [90]:
import numpy as np
from gurobipy import *
import pandas as pd
from collections import defaultdict
import itertools

# Hyperparamètres

In [91]:
# Hyperparamètres
acompte = 0.1  # Part d'acompte initial
duree_optimiste = 1 # 0 si durée totale prise en compte; 1 si différence
budget_annuel = 2e6

# Poids pour la fonction objectif
poids_2030 = 0
poids_2040 = 0
poids_2050 = 1

sobriete = 0.1

# Préparation des données
Récupération des matrices de gains, de coût, des listes des bâtiments, de leurs types et des durées de travaux.

In [92]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("dataset_efficacity_avec_duree.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "typo_dpe", "id_simulation", "conso_total_mwh_an"]]
)

# Comptage par catégorie
nb_buildings_by_type = (
    df_calibrated
    .groupby("typo_dpe")["building_name"]
    .count()
    .to_dict()
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================
df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# --- 2. Groupement ---
group_gains    = df_nc.groupby("building_name")["gains_totaux_mwh_an"].apply(list)
group_cost     = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)
group_duration = df_nc.groupby("building_name")["temps_de_travaux"].apply(list)  # en mois

# --- 3. Taille maximale ---
max_len = max(
    group_gains.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
gains_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_gains
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

group_type = (
    df_nc
    .groupby("building_name")["typo_dpe"]
    .first()
)

# --- 5. Métadonnées utiles ---
building_names = group_gains.index.tolist()
building_types = group_type.loc[building_names].tolist()

grouped = df_nc.groupby("building_name", sort=False)["id_simulation"].apply(list)
renovation_names_per_building = grouped.to_dict()

# Durée par simulation (en mois)
duration_by_sim = {
    row["id_simulation"]: int(row["temps_de_travaux"])
    for _, row in df_nc.iterrows()
}

df_calibrated = df_calibrated[
    df_calibrated["building_name"].isin(building_names)
]

## Construction du graphe de transitions (version originale)

On garde la même logique de compatibilité pour construire les transitions possibles.

In [ ]:
# ============================
# GRAPHE DE TRANSITIONS
# ============================

transition_graph = {}
# Colonnes travaux (G → N)
renov_cols = df.columns[6:14]

# --- Construction par bâtiment ---
for building, group in df_nc.groupby("building_name", sort=False):

    group = group.reset_index(drop=True)

    works = group[renov_cols].astype(bool).values
    costs = group["cout_investissement_euros"].values
    gains = group["gains_totaux_mwh_an"].values
    ids   = group["id_simulation"].tolist()
    durs  = group["temps_de_travaux"].astype(int).values  # durées en mois

    n = len(group)

    # ========================
    # MATRICE COMPATIBILITE
    # ========================
    compat = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if np.all(~works[j] | works[i]):
                compat[i, j] = 1

    # ========================
    # GENERATION DES TRANSITIONS
    # ========================
    transitions = []

    # état initial (aucun travaux)
    for i in range(n):
        transitions.append({
            "from": "none",
            "to":   ids[i],
            "cost": costs[i],
            "gain": gains[i],
            "duration": int(durs[i])
        })

    # transitions entre rénovations
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # j ⊂ i
            if compat[i, j] == 1 and compat[j, i] == 0:
                transitions.append({
                    "from": ids[j],
                    "to":   ids[i],
                    "cost": costs[i] - costs[j],
                    "gain": gains[i] - gains[j],
                    "duration": int(durs[i]) - int(durs[j]) if duree_optimiste == 1 else int(durs[i])
                })

    transition_graph[building] = transitions

# Isolation des travaux spécifiques (utile pour l'excel)
works_by_sim = {}
for _, row in df_nc.iterrows():
    works_by_sim[row["id_simulation"]] = {
        col: bool(row[col])
        for col in renov_cols
    }

In [94]:
# Vérification
example_building = list(transition_graph.keys())[0]
print(f"Bâtiment exemple: {example_building}")
print(f"Nombre de transitions: {len(transition_graph[example_building])}")
print("\nPremière transition:")
print(transition_graph[example_building][0])

Bâtiment exemple: conservatoire_maurice_bacquet
Nombre de transitions: 26

Première transition:
{'from': 'none', 'to': 'conservatoire_maurice_bacquet_simulation 1', 'cost': np.float64(6776.8), 'gain': np.float64(0.0958091433046206), 'duration': 2}


# Paramètres temporels

In [95]:
Conso_total  = np.sum(df_calibrated["conso_total_mwh_an"])
nbr_mois     = (2050 - 2026 + 1) * 12  # horizon en mois (t=0 → jan 2026, t=299 → déc 2050)
nbr_annees   = 2050 - 2026 + 1
nbr_batiments = gains_matrix.shape[0]

print("Conso total (avec sobriété 10%):", Conso_total, "MWh/an")
print("Nombre de bâtiments:", nbr_batiments)
print("Horizon:", nbr_mois, "mois")

Conso total (avec sobriété 10%): 20843.47743464322 MWh/an
Nombre de bâtiments: 71
Horizon: 300 mois


# Construction du Graphe Temporel (r, t)

## Concept

Pour chaque bâtiment, on construit un graphe temporel où:
- **Nœuds**: (état r, mois t)
- **Arêtes de 2 types**:
  1. **Attente**: (r, t) → (r, t+1) - coût 0, gain 0, durée 1
  2. **Transition**: (r, t) → (r', t+d) - selon les transitions du graphe original

## Variables de décision

- `f[b, e]` = 1 si l'arête e est empruntée dans le bâtiment b, 0 sinon

## Avantages

- Les contraintes de flux deviennent **locales** (par nœud)
- Plus besoin de contraintes cumulatives Σ_{τ≤t}
- Structure naturelle du problème de planification

In [96]:
# ============================
# CONSTRUCTION DU GRAPHE TEMPOREL
# ============================

temporal_graph = {}  # {building: {"nodes": [...], "edges": [...]}}

for b in building_names:
    
    # Extraire tous les états possibles pour ce bâtiment
    states = set(["none"])  # État initial
    for trans in transition_graph[b]:
        states.add(trans["from"])
        states.add(trans["to"])
    states.discard("none")  # On le rajoute après
    states = ["none"] + sorted(list(states))  # "none" en premier
    
    # Créer les nœuds (r, t)
    nodes = []
    node_index = {}  # (r, t) -> index
    
    for r in states:
        for t in range(nbr_mois + 1):  # +1 pour avoir un état final au-delà de l'horizon
            node_id = len(nodes)
            nodes.append({"state": r, "time": t, "id": node_id})
            node_index[(r, t)] = node_id
    
    # Créer les arêtes
    edges = []
    
    # Type 1: Arêtes d'attente (r, t) -> (r, t+1)
    for r in states:
        for t in range(nbr_mois):
            edges.append({
                "type": "wait",
                "from_node": node_index[(r, t)],
                "to_node": node_index[(r, t+1)],
                "from_state": r,
                "to_state": r,
                "from_time": t,
                "to_time": t + 1,
                "cost": 0,
                "gain": 0,
                "duration": 1,
                "original_transition_idx": None
            })
    
    # Type 2: Arêtes de transition (r, t) -> (r', t+d)
    # Basées sur le graphe de transitions original
    for trans_idx, trans in enumerate(transition_graph[b]):
        from_state = trans["from"]
        to_state = trans["to"]
        duration = trans["duration"]
        
        # On peut démarrer cette transition à n'importe quel moment t
        # tel que t + duration <= nbr_mois
        for t in range(nbr_mois - duration + 1):
            edges.append({
                "type": "transition",
                "from_node": node_index[(from_state, t)],
                "to_node": node_index[(to_state, t + duration)],
                "from_state": from_state,
                "to_state": to_state,
                "from_time": t,
                "to_time": t + duration,
                "cost": trans["cost"],
                "gain": trans["gain"],
                "duration": duration,
                "original_transition_idx": trans_idx
            })
    
    temporal_graph[b] = {
        "nodes": nodes,
        "edges": edges,
        "node_index": node_index,
        "states": states
    }

# Statistiques
total_nodes = sum(len(temporal_graph[b]["nodes"]) for b in building_names)
total_edges = sum(len(temporal_graph[b]["edges"]) for b in building_names)

print(f"Graphe temporel construit:")
print(f"  - Nombre total de nœuds: {total_nodes:,}")
print(f"  - Nombre total d'arêtes: {total_edges:,}")
print(f"\nPour le premier bâtiment '{building_names[0]}':")
print(f"  - États: {len(temporal_graph[building_names[0]]['states'])}")
print(f"  - Nœuds: {len(temporal_graph[building_names[0]]['nodes'])}")
print(f"  - Arêtes: {len(temporal_graph[building_names[0]]['edges'])}")

Graphe temporel construit:
  - Nombre total de nœuds: 257,656
  - Nombre total d'arêtes: 1,006,603

Pour le premier bâtiment 'abri_bouliste':
  - États: 16
  - Nœuds: 4816
  - Arêtes: 21641


# Modèle d'optimisation avec Gurobi

## Variables de décision

- `f[b, e]` = 1 si l'arête e est empruntée pour le bâtiment b, 0 sinon

## Contraintes

1. **Conservation du flux** : Pour chaque nœud (sauf source/puits), flux entrant = flux sortant
2. **Source unique** : Un seul flux sort du nœud ("none", 0)
3. **Puits unique** : Le flux arrive à un état final au temps T
4. **Budget annuel** : Contrainte de budget avec acompte/solde
5. **Disponibilité** : Au plus 20% des bâtiments d'une catégorie en transition simultanément
6. **Non-chevauchement** : Un bâtiment ne peut avoir qu'une transition active à la fois

In [97]:
m = Model("Efficacity - Graphe Temporel")

# Variables de décision: flux sur les arêtes
f = {}

for b in building_names:
    for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
        f[b, e_idx] = m.addVar(
            vtype=GRB.BINARY,
            name=f"f_{b}_{e_idx}"
        )

m.update()
print(f"Nombre de variables binaires : {m.NumVars:,}")

Nombre de variables binaires : 1,006,603


## Contrainte 1: Conservation du flux

Pour chaque nœud (r, t), sauf la source ("none", 0) et le puits:

$$\sum_{\text{arêtes entrantes}} f_e = \sum_{\text{arêtes sortantes}} f_e$$

In [98]:
# Précalculer les arêtes entrantes et sortantes pour chaque nœud
incoming_edges = {}  # {(b, node_id): [edge_indices]}
outgoing_edges = {}  # {(b, node_id): [edge_indices]}

for b in building_names:
    for node in temporal_graph[b]["nodes"]:
        node_id = node["id"]
        incoming_edges[b, node_id] = []
        outgoing_edges[b, node_id] = []
    
    for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
        outgoing_edges[b, edge["from_node"]].append(e_idx)
        incoming_edges[b, edge["to_node"]].append(e_idx)

# Contrainte de conservation du flux
for b in building_names:
    node_index = temporal_graph[b]["node_index"]
    
    # Source: ("none", 0) - exactement un flux sortant
    source_node = node_index[("none", 0)]
    m.addConstr(
        quicksum(f[b, e] for e in outgoing_edges[b, source_node]) == 1,
        name=f"source_{b}"
    )
    
    # Puits: tous les nœuds au temps nbr_mois - exactement un flux entrant
    sink_nodes = [node_index[(r, nbr_mois)] for r in temporal_graph[b]["states"]]
    m.addConstr(
        quicksum(
            f[b, e]
            for node_id in sink_nodes
            for e in incoming_edges[b, node_id]
        ) == 1,
        name=f"sink_{b}"
    )
    
    # Nœuds intermédiaires: conservation du flux
    for node in temporal_graph[b]["nodes"]:
        node_id = node["id"]
        state = node["state"]
        time = node["time"]
        
        # Ignorer la source et les puits
        if (state == "none" and time == 0) or time == nbr_mois:
            continue
        
        in_edges = incoming_edges[b, node_id]
        out_edges = outgoing_edges[b, node_id]
        
        if in_edges or out_edges:  # Seulement si le nœud a des connexions
            m.addConstr(
                quicksum(f[b, e] for e in in_edges) == 
                quicksum(f[b, e] for e in out_edges),
                name=f"flow_{b}_{node_id}"
            )

print(f"Contraintes de flux ajoutées")

Contraintes de flux ajoutées


## Contrainte 2: Budget annuel avec acompte/solde

Pour chaque année a:

$$\sum_{\text{transitions démarrées en a}} \text{acompte} \times \text{coût} + \sum_{\text{transitions finies en a}} (1-\text{acompte}) \times \text{coût} \leq \text{Budget}_a$$

In [99]:
def month_to_year(t):
    """Retourne l'index d'année (0 = 2026) correspondant au mois t."""
    return t // 12

for a in range(nbr_annees):
    # Mois appartenant à l'année a
    months_a = list(range(a * 12, (a + 1) * 12))
    
    # Acomptes payés (transitions démarrées en a)
    acomptes = []
    # Soldes payés (transitions finies en a)
    soldes = []
    
    for b in building_names:
        for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
            if edge["type"] != "transition":  # Pas de coût pour les arêtes d'attente
                continue
            
            start_time = edge["from_time"]
            end_time = edge["to_time"]
            cost = edge["cost"]
            
            # Acompte si la transition démarre en a
            if start_time in months_a:
                acomptes.append(acompte * cost * f[b, e_idx])
            
            # Solde si la transition se termine en a
            if end_time - 1 in months_a:  # -1 car end_time est exclusif
                soldes.append((1 - acompte) * cost * f[b, e_idx])
    
    if acomptes or soldes:
        m.addConstr(
            quicksum(acomptes) + quicksum(soldes) <= budget_annuel,
            name=f"budget_annee_{a}"
        )

print(f"Contraintes budgétaires ajoutées")

Contraintes budgétaires ajoutées


## Contrainte 3: Disponibilité par catégorie

Pour chaque catégorie et chaque mois t:

$$\sum_{\text{bâtiments de la catégorie}} \sum_{\text{transitions actives en t}} f_e \leq 0.2 \times |\text{catégorie}|$$

Une transition est **active** au mois t si:
- Elle démarre au mois τ
- Elle a une durée d
- τ ≤ t < τ + d

In [100]:
# Grouper les bâtiments par catégorie
buildings_by_cat = defaultdict(list)
for b_name, cat in zip(building_names, building_types):
    buildings_by_cat[cat].append(b_name)

# Mois exemptés pour la catégorie Enseignement (janvier=0, juillet=6, août=7 dans l'année)
EXEMPT_MONTHS_OF_YEAR = {0, 6, 7}  # indices dans l'année (0-based)
EXEMPT_CATEGORY = "Enseignement"

for cat, b_names in buildings_by_cat.items():
    if cat == "Autres":
        continue
    
    card_cat = len(b_names)
    
    for t in range(nbr_mois):
        # Vérifier si ce mois est exempté pour la catégorie Enseignement
        month_in_year = t % 12
        is_exempt_month = month_in_year in EXEMPT_MONTHS_OF_YEAR
        
        if cat == EXEMPT_CATEGORY and is_exempt_month:
            # Pas de contrainte de disponibilité pour Enseignement en jan/jul/août
            continue
        
        # Compter les transitions actives au mois t
        active_transitions = []
        
        for b in b_names:
            for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
                if edge["type"] != "transition":
                    continue
                
                start_time = edge["from_time"]
                end_time = edge["to_time"]
                
                # La transition est active si start_time <= t < end_time
                if start_time <= t < end_time:
                    active_transitions.append(f[b, e_idx])
        
        if active_transitions:
            m.addConstr(
                quicksum(active_transitions) <= 0.2 * card_cat,
                name=f"availability_{cat}_{t}"
            )

print(f"Contraintes de disponibilité ajoutées")

Contraintes de disponibilité ajoutées


## Contrainte 4: Non-chevauchement des transitions

Pour chaque bâtiment et chaque mois t, au plus une transition peut être active:

$$\sum_{\text{transitions actives en t}} f_e \leq 1$$

Cette contrainte est **redondante** avec la structure du graphe temporel (conservation du flux), mais on l'ajoute pour renforcer le modèle.

In [101]:
for b in building_names:
    for t in range(nbr_mois):
        active_transitions = []
        
        for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
            if edge["type"] != "transition":
                continue
            
            start_time = edge["from_time"]
            end_time = edge["to_time"]
            
            # La transition est active si start_time <= t < end_time
            if start_time <= t < end_time:
                active_transitions.append(f[b, e_idx])
        
        if active_transitions:
            m.addConstr(
                quicksum(active_transitions) <= 1,
                name=f"no_overlap_{b}_{t}"
            )

print(f"Contraintes de non-chevauchement ajoutées")

Contraintes de non-chevauchement ajoutées


## Fonction objectif

Maximiser la somme pondérée des gains réalisés avant 2030, 2040 et 2050.

Un gain est **réalisé** lorsque la transition est **terminée**, c'est-à-dire au temps `to_time`.

$$\max \quad 0.25 \times \text{gains}_{2030} + 0.25 \times \text{gains}_{2040} + 0.5 \times \text{gains}_{2050}$$

In [102]:
# Dernier mois inclus dans chaque période (fin décembre)
T_2030 = (2030 - 2026 + 1) * 12 - 1   # = 59  (décembre 2030)
T_2040 = (2040 - 2026 + 1) * 12 - 1   # = 179 (décembre 2040)
T_2050 = nbr_mois - 1                  # = 299 (décembre 2050)

def gain_before(T):
    """Somme des gains dont la fin de rénovation (to_time - 1) est <= T."""
    terms = []
    for b in building_names:
        for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
            if edge["type"] != "transition":
                continue
            
            # La transition se termine au mois to_time - 1
            end_time = edge["to_time"] - 1
            
            if end_time <= T:
                terms.append(edge["gain"] * f[b, e_idx])
    
    return quicksum(terms) if terms else 0

m.setObjective(
      poids_2030 * gain_before(T_2030)
    + poids_2040 * gain_before(T_2040)
    + poids_2050 * gain_before(T_2050),
    GRB.MAXIMIZE
)

print("Fonction objectif définie")

Fonction objectif définie


In [103]:
m.update()
print(f"Nombre de variables du modèle: {m.NumVars:,}")
print(f"Nombre de contraintes du modèle: {m.NumConstrs:,}")

Nombre de variables du modèle: 1,006,603
Nombre de contraintes du modèle: 279,021


# Résolution du modèle

In [104]:
# Paramètres de résolution
m.setParam('TimeLimit', 5 * 60)  # 5 minutes
m.setParam('MIPGap', 0.01)       # Gap de 1%
m.setParam('OutputFlag', 1)      # Afficher les logs

m.update()
print("Démarrage de l'optimisation...")
m.optimize()

Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.01
Set parameter OutputFlag to value 1
Démarrage de l'optimisation...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 279021 rows, 1006603 columns and 4915903 nonzeros (Max)
Model fingerprint: 0x0a6b0d5a
Model has 748608 linear objective coefficients
Variable types: 0 continuous, 1006603 integer (1006603 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+06]
  Objective range  [1e-05, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]

Presolve removed 81066 rows and 92847 columns (presolve time = 5s)...
Presolve removed 81765 rows and 125480 columns (presolve time = 10s)...
Presolve removed 81810 rows 

# Analyse des résultats

In [105]:
if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
    print(f"\nStatut: {'OPTIMAL' if m.status == GRB.OPTIMAL else 'TIME_LIMIT'}")
    print(f"Valeur objectif : {m.ObjVal:.4f}")
    if m.status == GRB.TIME_LIMIT:
        print(f"Gap MIP : {m.MIPGap * 100:.2f}%")
else:
    print(f"\nProblème de résolution. Statut: {m.status}")


Statut: OPTIMAL
Valeur objectif : 9419.2403


## Calcul des objectifs réalisés (avec sobriété 10%)

In [106]:
def realized_gain_before(T):
    """Calcule les gains réalisés avant le temps T."""
    total_gain = 0
    for b in building_names:
        for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
            if edge["type"] != "transition":
                continue
            
            if f[b, e_idx].X > 0.5:  # Arête empruntée
                end_time = edge["to_time"] - 1
                if end_time <= T:
                    total_gain += edge["gain"]
    return total_gain

if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
    obj_2030 = sobriete + (1 - sobriete) * realized_gain_before(T_2030) / Conso_total
    obj_2040 = sobriete + (1 - sobriete) * realized_gain_before(T_2040) / Conso_total
    obj_2050 = sobriete + (1 - sobriete) * realized_gain_before(T_2050) / Conso_total

    print(f"\nObjectifs réalisés:")
    print(f"Objectif 2030 : {obj_2030:.4f} (cible 0.4)")
    print(f"Objectif 2040 : {obj_2040:.4f} (cible 0.5)")
    print(f"Objectif 2050 : {obj_2050:.4f} (cible 0.6)")


Objectifs réalisés:
Objectif 2030 : 0.2005 (cible 0.4)
Objectif 2040 : 0.3095 (cible 0.5)
Objectif 2050 : 0.5067 (cible 0.6)


## Extraction du chemin suivi par chaque bâtiment

In [107]:
if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
    paths = {}  # {building: [(from_state, to_state, from_time, to_time, type)]}
    
    for b in building_names:
        paths[b] = []
        for e_idx, edge in enumerate(temporal_graph[b]["edges"]):
            if f[b, e_idx].X > 0.5:  # Arête empruntée
                paths[b].append({
                    "from_state": edge["from_state"],
                    "to_state": edge["to_state"],
                    "from_time": edge["from_time"],
                    "to_time": edge["to_time"],
                    "type": edge["type"],
                    "cost": edge["cost"],
                    "gain": edge["gain"]
                })
        
        # Trier par temps de départ
        paths[b].sort(key=lambda x: x["from_time"])
    
    # Afficher quelques exemples
    print("\nExemples de chemins suivis:\n")
    for i, b in enumerate(building_names[:3]):
        print(f"Bâtiment: {b}")
        for step in paths[b]:
            if step["type"] == "transition":
                print(f"  Mois {step['from_time']:3d}-{step['to_time']:3d}: "
                      f"{step['from_state']:30s} → {step['to_state']:30s} "
                      f"(gain: {step['gain']:6.2f} MWh/an, coût: {step['cost']:10.2f} €)")
        print()


Exemples de chemins suivis:

Bâtiment: abri_bouliste
  Mois 122-123: none                           → abri_bouliste_simulation_11    (gain:   6.09 MWh/an, coût:   62094.12 €)

Bâtiment: accueil_de_loisirs_maternel_la_varenne
  Mois 175-177: none                           → accueil_de_loisirs_maternel_la_varenne_simulation_14 (gain:   3.32 MWh/an, coût:  241467.91 €)

Bâtiment: association_31_rue_gambetta
  Mois 262-264: none                           → association_31_rue_gambetta_simulation_2 (gain:   7.08 MWh/an, coût:   74048.48 €)
  Mois 275-275: association_31_rue_gambetta_simulation_2 → association_31_rue_gambetta_simulation_6 (gain:   0.24 MWh/an, coût:   93003.02 €)



## Analyse des transitions réalisées

In [108]:
if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
    # Statistiques globales
    total_transitions = 0
    total_cost = 0
    total_gain = 0
    
    transitions_by_year = defaultdict(int)
    cost_by_year = defaultdict(float)
    
    for b in building_names:
        for step in paths[b]:
            if step["type"] == "transition":
                total_transitions += 1
                total_cost += step["cost"]
                total_gain += step["gain"]
                
                # Année de démarrage
                year = 2026 + step["from_time"] // 12
                transitions_by_year[year] += 1
                cost_by_year[year] += step["cost"]
    
    print(f"\nStatistiques globales:")
    print(f"  - Nombre total de transitions: {total_transitions}")
    print(f"  - Coût total: {total_cost:,.2f} €")
    print(f"  - Gain total: {total_gain:,.2f} MWh/an")
    print(f"  - Bâtiments rénovés: {sum(1 for b in building_names if any(s['type'] == 'transition' for s in paths[b]))} / {nbr_batiments}")
    
    print(f"\nRépartition par année:")
    for year in sorted(transitions_by_year.keys()):
        print(f"  {year}: {transitions_by_year[year]:2d} transitions, {cost_by_year[year]:10,.2f} € (budget: {budget_annuel:,.2f} €)")


Statistiques globales:
  - Nombre total de transitions: 127
  - Coût total: 48,809,717.32 €
  - Gain total: 9,419.24 MWh/an
  - Bâtiments rénovés: 71 / 71

Répartition par année:
  2026:  7 transitions, 4,317,086.05 € (budget: 2,000,000.00 €)
  2027: 11 transitions, 1,162,002.99 € (budget: 2,000,000.00 €)
  2028:  3 transitions, 601,286.10 € (budget: 2,000,000.00 €)
  2029:  5 transitions, 3,703,038.21 € (budget: 2,000,000.00 €)
  2030:  1 transitions, 848,130.00 € (budget: 2,000,000.00 €)
  2031:  7 transitions, 2,335,243.33 € (budget: 2,000,000.00 €)
  2032:  5 transitions, 1,936,137.56 € (budget: 2,000,000.00 €)
  2033:  5 transitions, 2,330,139.26 € (budget: 2,000,000.00 €)
  2034:  2 transitions, 280,425.71 € (budget: 2,000,000.00 €)
  2035:  4 transitions, 1,914,681.36 € (budget: 2,000,000.00 €)
  2036:  6 transitions, 1,955,968.01 € (budget: 2,000,000.00 €)
  2037:  3 transitions, 1,892,354.92 € (budget: 2,000,000.00 €)
  2038:  2 transitions, 1,995,573.00 € (budget: 2,000,000.

## Export des résultats

In [109]:
if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
    
    # =========================
    # Fonction auxiliaire pour identifier les travaux
    # =========================
    def works_added(from_state, to_state):
        """Isolation des travaux spécifiques à partir d'une arête du graphe.
        Utile pour connaître exactement quelles rénovations sont faites."""
        if from_state == "none":
            return [
                col for col, v in works_by_sim[to_state].items() if v
            ]
        
        return [
            col
            for col in renov_cols
            if works_by_sim[to_state][col] and not works_by_sim[from_state][col]
        ]
    
    # =========================
    # Paramètres temporels
    # =========================
    start_year = 2026
    end_year = 2050
    nbr_years = end_year - start_year + 1
    nbr_months_total = 12 * nbr_years
    
    month_labels = [
        f"{start_year + t // 12}-{(t % 12) + 1:02d}"
        for t in range(nbr_months_total)
    ]
    
    year_labels = list(range(start_year, end_year + 1))
    
    def month_to_year_month_period(t):
        """Convertit un indice mensuel t en (year, month, period)."""
        year = start_year + t // 12
        month = (t % 12) + 1
        period = f"{year}-{month:02d}"
        return year, month, period
    
    # =========================
    # Créer un DataFrame avec toutes les transitions
    # =========================
    transitions_data = []
    
    for b in building_names:
        for step in paths[b]:
            if step["type"] == "transition":
                year, month, period = month_to_year_month_period(step["from_time"])
                added = works_added(step["from_state"], step["to_state"])
                
                transitions_data.append({
                    "building": b,
                    "month_index": step["from_time"],
                    "year": year,
                    "month": month,
                    "period": period,
                    "from_state": step["from_state"],
                    "to_state": step["to_state"],
                    "transition_label": f'{step["from_state"]} -> {step["to_state"]}',
                    "works_added": ", ".join(added),
                    "start_month": step["from_time"],
                    "end_month": step["to_time"],
                    "start_year": 2026 + step["from_time"] // 12,
                    "end_year": 2026 + (step["to_time"] - 1) // 12,
                    "duration_months": step["to_time"] - step["from_time"],
                    "cost": step["cost"],
                    "gain": step["gain"]
                })
    
    df_transitions = pd.DataFrame(transitions_data)
    
    # =========================
    # Timeline des rénovations (noms des transitions)
    # =========================
    if not df_transitions.empty:
        timeline_renovations = (
            df_transitions
            .pivot_table(
                index="building",
                columns="period",
                values="transition_label",
                aggfunc="first"
            )
            .reindex(columns=month_labels)
        )
        
        # Timeline des travaux spécifiques
        timeline_works = (
            df_transitions
            .groupby(["building", "period"])["works_added"]
            .agg(lambda x: " | ".join(sorted(set(v for v in x if pd.notna(v) and v != ""))))
            .unstack()
            .reindex(columns=month_labels)
        )
        
        # Timeline des états (to_state)
        timeline_to_state = (
            df_transitions
            .pivot_table(
                index="building",
                columns="period",
                values="to_state",
                aggfunc="first"
            )
            .reindex(columns=month_labels)
        )
    else:
        timeline_renovations = pd.DataFrame()
        timeline_works = pd.DataFrame()
        timeline_to_state = pd.DataFrame()
    
    # =========================
    # Statistiques par année avec budget CORRECT
    # =========================
    stats_annees = []
    
    for a in range(nbr_annees):
        year = 2026 + a
        months_a = list(range(a * 12, (a + 1) * 12))
        
        # Calculer le budget RÉELLEMENT payé en année a
        budget_start_paid = 0  # Acomptes des transitions démarrées en a
        budget_end_paid = 0    # Soldes des transitions finies en a
        
        for b in building_names:
            for step in paths[b]:
                if step["type"] != "transition":
                    continue
                
                start_time = step["from_time"]
                end_time = step["to_time"]
                cost = step["cost"]
                
                # Acompte si la transition démarre en a
                if start_time in months_a:
                    budget_start_paid += acompte * cost
                
                # Solde si la transition se termine en a
                if (end_time - 1) in months_a:  # -1 car end_time est exclusif
                    budget_end_paid += (1 - acompte) * cost
        
        cost_total = budget_start_paid + budget_end_paid
        
        stats_annees.append({
            "year": year,
            "renovations_started": transitions_by_year.get(year, 0),
            "budget_acompte": budget_start_paid,
            "budget_solde": budget_end_paid,
            "budget_total_paye": cost_total,
        })
    
    df_stats_annees = pd.DataFrame(stats_annees)
    
    # =========================
    # Statistiques par mois
    # =========================
    if not df_transitions.empty:
        all_months_index = pd.MultiIndex.from_tuples(
            [
                (
                    start_year + t // 12,
                    (t % 12) + 1,
                    f"{start_year + t // 12}-{(t % 12) + 1:02d}"
                )
                for t in range(nbr_months_total)
            ],
            names=["year", "month", "period"]
        )
        
        stats_months = (
            df_transitions
            .groupby(["year", "month", "period"])
            .agg(
                renovations_started=("building", "count"),
                gain_total=("gain", "sum"),
                cost_total_started=("cost", "sum")
            )
            .reindex(all_months_index, fill_value=0)
            .reset_index()
        )
    else:
        stats_months = pd.DataFrame(columns=["year", "month", "period", "renovations_started", "gain_total", "cost_total_started"])
    
    # =========================
    # Export Excel
    # =========================
    with pd.ExcelWriter("plan_renovation_graphe_temporel.xlsx") as writer:
        # Feuille 1: Transitions détaillées
        df_transitions.to_excel(writer, sheet_name="transitions", index=False)
        
        # Feuille 2: Timeline des rénovations
        if not timeline_renovations.empty:
            timeline_renovations.to_excel(writer, sheet_name="timeline_renovations")
        
        # Feuille 3: Timeline des travaux spécifiques
        if not timeline_works.empty:
            timeline_works.to_excel(writer, sheet_name="timeline_travaux")
        
        # Feuille 4: Timeline des états finaux
        if not timeline_to_state.empty:
            timeline_to_state.to_excel(writer, sheet_name="timeline_to_state")
        
        # Feuille 5: Statistiques mensuelles
        stats_months.to_excel(writer, sheet_name="stats_mois", index=False)
        
        # Feuille 6: Statistiques annuelles
        df_stats_annees.to_excel(writer, sheet_name="stats_annees", index=False)
    
    print("\nRésultats exportés dans 'plan_renovation_graphe_temporel.xlsx'")
    print("\nFeuilles Excel créées:")
    print("  1. transitions: Liste complète des transitions avec détails")
    print("  2. timeline_renovations: Timeline des noms de rénovations par bâtiment")
    print("  3. timeline_travaux: Timeline des travaux spécifiques par bâtiment")
    print("  4. timeline_to_state: Timeline des états finaux atteints")
    print("  5. stats_mois: Statistiques mensuelles")
    print("  6. stats_annees: Statistiques annuelles avec budget correct (acompte + solde)")

PermissionError: [Errno 13] Permission denied: 'plan_renovation_graphe_temporel.xlsx'

# Comparaison avec l'ancienne formulation

## Nombre de variables

- **Ancienne formulation** : ~750 000 variables
- **Nouvelle formulation** : Varie selon le nombre d'arêtes

## Nombre de contraintes

- **Ancienne formulation** : ~258 000 contraintes (dont 150k de flux cumulatifs)
- **Nouvelle formulation** : Contraintes de flux locales (une par nœud interne)

## Avantages attendus

1. **Contraintes plus simples** : Flux locaux au lieu de cumulatifs
2. **Structure plus naturelle** : Le graphe temporel reflète mieux le problème
3. **Meilleure interprétabilité** : Le chemin suivi est explicite
4. **Potentiellement moins de sur-contrainte** : Les contraintes cumulatives pouvaient bloquer certaines solutions

## Points d'attention

- Le nombre d'arêtes peut être très grand (nombre d'états × nombre de mois × transitions possibles)
- Il faut vérifier que le modèle ne devient pas trop gros en mémoire

In [ ]:
print("=" * 60)
print("COMPARAISON DES FORMULATIONS")
print("=" * 60)
print(f"\nAncienne formulation:")
print(f"  - Variables: ~750,000")
print(f"  - Contraintes: ~258,000 (dont ~150k cumulatives)")
print(f"\nNouvelle formulation (graphe temporel):")
print(f"  - Variables: {m.NumVars:,}")
print(f"  - Contraintes: {m.NumConstrs:,}")
print(f"  - Nœuds du graphe: {total_nodes:,}")
print(f"  - Arêtes du graphe: {total_edges:,}")

COMPARAISON DES FORMULATIONS

Ancienne formulation:
  - Variables: ~750,000
  - Contraintes: ~258,000 (dont ~150k cumulatives)

Nouvelle formulation (graphe temporel):
  - Variables: 1,002,034
  - Contraintes: 279,021
  - Nœuds du graphe: 257,656
  - Arêtes du graphe: 1,002,034
